# Explorative Analyse und Data Proprocessing:

In [ ]:
import pandas as pd
import matplotlib
import numpy as np
#%matplotlib inline

In [ ]:
#Zusammenstellen eines Dataframe mit allen ausgewählten Zählern
df3310 = pd.read_csv('../data/bzm_telraam_9000003310.csv.gz.csv')
df4118 = pd.read_csv('../data/bzm_telraam_9000004118.csv.gz.csv')
df4602 = pd.read_csv('../data/bzm_telraam_9000004602.csv.gz.csv')
df4685 = pd.read_csv('../data/bzm_telraam_9000004685.csv.gz.csv')
df5444 = pd.read_csv('../data/bzm_telraam_9000005444.csv.gz.csv')
df5832 = pd.read_csv('../data/bzm_telraam_9000005832.csv.gz.csv')

df_zaehler= pd.concat([df3310, df4118, df4602, df4685, df5444, df5832], ignore_index=True)

In [ ]:
#Anzeige der ersten Zeilen
df_zaehler.head()

In [ ]:
# Überprüfen der Plausibilität von ped_total

df_zaehler["summe_ped"]=df_zaehler["ped_lft"]+df_zaehler["ped_rgt"]
df_zaehler["summe_ped_valid"]=df_zaehler["ped_total"]==df_zaehler["summe_ped"]
plausibel_false_ped = (df_zaehler["summe_ped_valid"]==False).sum()
print(plausibel_false_ped)


df_notPlausible_ped=df_zaehler[df_zaehler["summe_ped_valid"]==False]
df_notPlausible_ped_copy= df_notPlausible_ped.copy()
df_notPlausible_ped_copy["ped_total_fehler"]=df_notPlausible_ped["ped_total"]-df_notPlausible_ped["summe_ped"]
df_notPlausible_ped_copy.describe()



In [ ]:
# Überprüfen der Plausibilität von bike_total
df_zaehler["summe_bike"]=df_zaehler["bike_lft"]+df_zaehler["bike_rgt"]
df_zaehler["summe_bike_valid"]=df_zaehler["bike_total"]==df_zaehler["summe_bike"]
plausibel_false_bike = (df_zaehler["summe_bike_valid"]==False).sum()
print(plausibel_false_bike)

df_notPlausible_bike=df_zaehler[df_zaehler["summe_bike_valid"]==False]
df_notPlausible_bike["bike_total_fehler"]=df_notPlausible_bike["bike_total"]-df_notPlausible_bike["summe_bike"]
df_notPlausible_bike.describe()

In [ ]:
# Überprüfen der Plausibilität von car_total

df_zaehler["summe_car"]=df_zaehler["car_lft"]+df_zaehler["car_rgt"]
df_zaehler["summe_car_valid"]=df_zaehler["car_total"]==df_zaehler["summe_car"]
plausibel_false_car = (df_zaehler["summe_car_valid"]==False).sum()
print(plausibel_false_car)

df_notPlausible_car=df_zaehler[df_zaehler["summe_car_valid"]==False]
df_notPlausible_car["car_total_fehler"]=df_notPlausible_car["car_total"]-df_notPlausible_car["summe_car"]
df_notPlausible_car.describe()

In [ ]:
# Stündliche Werte für ped_total, wie unter Datengrundlage - Telraam erklärt 
df_zaehler['pedtotal_corrected'] = (df_zaehler['ped_total'] * (1 - df_zaehler['uptime'])) + df_zaehler['ped_total']

In [ ]:
# Stündliche Werte für car_total, wie unter Datengrundlage - Telraam erklärt 
df_zaehler['cartotal_corrected'] = (df_zaehler['car_total'] * (1 - df_zaehler['uptime'])) + df_zaehler['car_total']

In [ ]:
# Stündliche Werte für bike_total, wie unter Datengrundlage - Telraam erklärt 
df_zaehler['biketotal_corrected'] = (df_zaehler['bike_total'] * (1 - df_zaehler['uptime'])) + df_zaehler['bike_total']

In [ ]:
#Wir haben einen hohen Wert rausgeworfen. 
df_zaehler=df_zaehler.drop(df_zaehler[df_zaehler['ped_total'] == 1800].index)

In [ ]:
#Datum umwandeln in datetime Format
df_zaehler["date_local"] = pd.to_datetime (df_zaehler["date_local"], format="%Y-%m-%d %H:%M")
                                                    

In [ ]:
df_zaehler.head()

In [ ]:
#Daten rausfiltern, die weniger als 30 Minuten Datenaufnahme pro Stunde haben
df_zaehler = df_zaehler.drop(df_zaehler[df_zaehler["uptime"]<= 0.5].index)
#Daten rausfiltern, in denen uptime mehr als 100 % ist
df_zaehler = df_zaehler.drop(df_zaehler[df_zaehler["uptime"]> 1].index)

In [ ]:
# Graph der Zielvariable "pedtotal_corrected" von allen Zählern 
from matplotlib import pyplot as plt

plt.figure(figsize=(10, 5))
for zaehler, variable in df_zaehler.groupby('segment_id'):
    plt.plot(variable['date_local'], variable['pedtotal_corrected'], marker='o', label=zaehler)


plt.title('Fußgänger*innen pro Stunde je Zähler')
plt.xlabel('Datum')
plt.ylabel('Fußgänger*innen pro Stunde')
plt.grid(True)
plt.xticks(rotation=45)
#plt.savefig("../images/ProStundeJeZaehler.jpg", format='jpg')
plt.tight_layout()

In [ ]:
# Graph für jeden Zähler einzelnd
df_zaehlerId = df_zaehler['segment_id'].unique()
anzahl_zaehlerId = len(df_zaehlerId)

In [ ]:
# Erstellen der Subplots
plt.figure(figsize=(10, anzahl_zaehlerId * 3))  # Die Größe des Gesamtplots anpassen

# Iteriere über die Zähler und erstelle für jeden einen eigenen Plot
for i, (zaehler, variable) in enumerate(df_zaehler.groupby('segment_id')):
    plt.subplot(anzahl_zaehlerId, 1, i + 1)  # Platzierung des Subplots
    plt.plot(variable['date_local'], variable['pedtotal_corrected'], marker='o', label=f'segment_id {zaehler}')

    plt.title(f'Zählernummer: {zaehler}')
    plt.xlabel('Datum')
    plt.ylabel('Anzahl Fußgänger*innen pro Stunde')
    plt.grid(True)
    plt.xticks(rotation=45)

plt.tight_layout()  # Für besseren Abstand zwischen den Subplots
#plt.savefig("../images/SubplotZaehler.jpg", format='jpg')
plt.show()

In [ ]:
#Zeilen und Spaltenanzahl des Dataframe
df_zaehler.shape

In [ ]:
#Datentypen im Datensatz
df_zaehler.dtypes

In [ ]:
#Statistischer Überblick über Datensatz
df_zaehler.describe()

In [ ]:
#Fehlende Werte im Datensatz werden angezeigt. V85 steht für die Geschwindigkeit der Autos und interessiert uns hier nicht.
df_zaehler.isna().sum()

In [ ]:
#Einlesen der stündlichen Wetterdaten
df_regen = pd.read_csv('../data/produkt_rr_stunde_19950901_20241231_00433.txt',sep=';')
df_temp = pd.read_csv('../data/produkt_tu_stunde_19510101_20241231_00433.txt',sep=';')

In [ ]:
df_regen.head()

In [ ]:
df_temp.head()


In [ ]:
df_regen.describe()

In [ ]:
df_temp.describe()

In [ ]:
#Mess_datum in ein Datum umwandeln
df_regen["MESS_DATUM"] = pd.to_datetime(df_regen["MESS_DATUM"],format="%Y%m%d%H") #,format="%Y%m%d%H" hiermit wird das Datum umgewandelt, damit es in Zukunft einfacher zu handhaben ist. Das Format ist JahrMonatTag, deswegen %Y%md%Y%m%d
df_temp["MESS_DATUM"] = pd.to_datetime(df_temp["MESS_DATUM"], format="%Y%m%d%H")#

In [ ]:
df_wetter = pd.merge(
    df_regen[["MESS_DATUM", "  R1"]],
    df_temp[["MESS_DATUM", "TT_TU"]],
    on=["MESS_DATUM"],
    how="inner" #inner schmeißt alles raus was nicht in beiden DataFrames steht
)

In [ ]:
df_wetter.head()

In [ ]:
# Umwandeln von Messdatum in Mikrosekunden (nicht notwendig zum mergen)
#df_wetter["MESS_DATUM"]=df_wetter["MESS_DATUM"].astype('datetime64[us]')

In [ ]:
#Datentypen im Datensatz
df_wetter.dtypes

In [ ]:
df_wetter.head()

In [ ]:
df_wetter.isna().sum()

In [ ]:
df_wetter.describe()


In [ ]:
df_modellvariablen = pd.merge(
    df_zaehler,
    df_wetter, #[["  R1", "MESS_DATUM", "TT_TU"]]
    left_on=["date_local"],
    right_on=["MESS_DATUM"],
    how="inner" #inner schmeißt alles raus was nicht in beiden DataFrames steht
)

In [ ]:
# nicht notwendig - merge funktioniert
# print (df_zaehler["date_local"].dt.tz)

In [ ]:
df_modellvariablen.describe()

In [ ]:
df_modellvariablen.shape

In [ ]:
df_zaehler.shape

In [ ]:
print(df_modellvariablen.columns)

In [ ]:
countminus= (df_modellvariablen["  R1"]==-999.000000).sum()

In [ ]:
countminus2= (df_modellvariablen["TT_TU"]==-999.000000).sum()

In [ ]:
countminus

In [ ]:
countminus2

In [ ]:
# -999 Werte von Temperatur und Niederschlag durch NaN ersetzen und mit Median füllen 
df_modellvariablen["  R1"] = df_modellvariablen["  R1"].replace(-999, np.nan).fillna(df_modellvariablen["  R1"].median())
df_modellvariablen["TT_TU"] = df_modellvariablen["TT_TU"].replace(-999, np.nan).fillna(df_modellvariablen["TT_TU"].median())


In [ ]:
# R1 und TT_TU umbenennen
df_modellvariablen=df_modellvariablen.rename(columns={"  R1": "niederschlag", "TT_TU" : "temperatur" })


In [ ]:
df_modellvariablen.describe()

In [ ]:
df_modellvariablen.head()

  ### Variablen über datetime hinzufügen

In [ ]:

df_modellvariablen["stunde"]= df_modellvariablen["date_local"].dt.hour 
df_modellvariablen["DOY"]=df_modellvariablen["date_local"].dt.day_of_year # DOY = Tag des Jahres (bis auf die Stunde genau)

In [ ]:
df_modellvariablen["tag"] = df_zaehler["date_local"].dt.day
df_modellvariablen["monat"] = df_zaehler["date_local"].dt.month
df_modellvariablen["jahr"] = df_zaehler["date_local"].dt.year

In [ ]:
# Wochentag Feature hinzufügen (0=Montag, 6=Sonntag) 
df_modellvariablen["wochentag"] = df_modellvariablen["date_local"].dt.dayofweek 
df_modellvariablen["wochenende"] = (df_modellvariablen["wochentag"] >= 5).astype(int) # zu 0 oder 1 -> Kategorie




In [ ]:
tage=["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]
for i in range (7):
    df_modellvariablen[tage[i]]=(df_modellvariablen["wochentag"] == i).astype(float)

In [ ]:
df_modellvariablen.head()

### Nichtgenutzte Variablen löschen todo

## Übersicht Datensatz mit Modellvariablen

In [ ]:
pd.set_option('display.max_columns', None)
df_modellvariablen.head()

In [ ]:
print(df_modellvariablen.columns)

# Modelling

## Testsplit für alle Modelle:

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
print(df_modellvariablen.columns)

In [ ]:
# Features (X) und Ziel-Variable (y) definieren
X = df_modellvariablen[["niederschlag", "temperatur", "DOY", "stunde", "wochentag", "wochenende", "biketotal_corrected", "cartotal_corrected", "Mo", "Di", "Mi", "Do", "Fr", "Sa", "So","car_speed10", "car_speed20", "car_speed30", "car_speed40",
       "car_speed50", "car_speed60", "car_speed70" ]]  # Wetterdaten als Features
y = df_modellvariablen["pedtotal_corrected"]  # Fußgängerinnen als Ziel-Variable

In [ ]:
# Train/Test Split (80% Training, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Baseline Modell: Mittelwert 

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
#Evaluation des BaselineModells:
baselinemodell_true_test=y_test
baselinemodell_true_train=y_train
baselinemodel_pred_train=np.full(baselinemodell_true_train.shape, y_train.mean())
baselinemodel_pred_test=np.full(baselinemodell_true_test.shape, y_test.mean())



In [ ]:
modell_baseline = "Baselinemodell"
baseline_mae_train=mean_absolute_error(baselinemodell_true_train, baselinemodel_pred_train)
baseline_rmse_train=np.sqrt(mean_squared_error(baselinemodell_true_train, baselinemodel_pred_train))
baseline_r2_train=r2_score(baselinemodell_true_train, baselinemodel_pred_train)
baseline_mae_test=mean_absolute_error(baselinemodell_true_test, baselinemodel_pred_test)
baseline_rmse_test=np.sqrt(mean_squared_error(baselinemodell_true_test, baselinemodel_pred_test))
baseline_r2_test=r2_score(baselinemodell_true_test, baselinemodel_pred_test)


In [ ]:
ergebnis_baseline=[
    {"Modell" : modell_baseline, "Metrik": "MAE", "Training" : baseline_mae_train, "Test":baseline_mae_test},
    {"Modell" : modell_baseline, "Metrik": "RMSE", "Training" : baseline_rmse_train,  "Test":baseline_rmse_test},
    {"Modell" : modell_baseline, "Metrik": "R²", "Training" : baseline_r2_train,  "Test":baseline_r2_test}
     ]
ergebnis_baseline_df=pd.DataFrame(ergebnis_baseline)

In [ ]:
print(ergebnis_baseline_df.to_string(index=False))

In [ ]:
#todo: weglassen?
print(f"{'Metrik':<15} {'Training':<15} {'Test':<15}")
print(f"{'MAE':<15} {baseline_mae_train:<15.2f} {baseline_mae_test:<15.2f}")
print(f"{'RMSE':<15} {baseline_rmse_train:<15.2f} {baseline_rmse_test:<15.2f}")
print(f"{'R²':<15} {baseline_r2_train:<15.4f} {baseline_r2_test:<15.4f}")

## Lineares Regressionsmodell

In [ ]:
#Lineare Regression:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler



In [ ]:
# Daten normalisieren mit StandardScaler
#fit transform: Fittet Standardscaler auf die Daten und wendet die Transformation auf die Daten an
# transform: nutzt dieselbe skalierung und wendet die Transformation auf die Daten an
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Lineares Regressionsmodell trainieren
model_linreg = LinearRegression()
model_linreg.fit(X_train_scaled, y_train)

In [ ]:
# Koeffizienten anzeigen / Gewichtungen
print("Modell-Koeffizienten:")
for feature, coef in zip(["niederschlag", "temperatur", "DOY", "stunde", "wochentag", "wochenende", "biketotal_corrected", "cartotal_corrected", "Mo", "Di", "Mi", "Do", "Fr", "Sa", "So","car_speed10", "car_speed20", "car_speed30", "car_speed40",
       "car_speed50", "car_speed60", "car_speed70"], model_linreg.coef_):
    #"niederschlag", "temperatur", "DOY", "stunde", "wochentag", "wochenende", "biketotal_corrected", "cartotal_corrected"
    print(f"{feature:10s}: {coef:10.4f}")
print(f"Intercept: {model_linreg.intercept_:10.4f}")

In [ ]:
#wenn die Stunden sinkt die Anzahl der Fußganegerinnen
#Wenn die Niederschlagsmenge steigt, dann sinkt die Anzahl der Fußgängerinnen
#Intercept = Achsenabschnitt

In [ ]:
# Vorhersagen machen
y_pred_train_linearr = model_linreg.predict(X_train_scaled)
y_pred_test_linearr = model_linreg.predict(X_test_scaled)


In [ ]:
# Modell Lineare Regression evaluieren
modell_linearr="Lineare Regression"
train_mae_linearr = mean_absolute_error(y_train, y_pred_train_linearr)
train_rmse_linarr = np.sqrt(mean_squared_error(y_train, y_pred_train_linearr))
train_r2_linearr = r2_score(y_train, y_pred_train_linearr)

test_mae_linearr = mean_absolute_error(y_test, y_pred_test_linearr)
test_rmse_linearr = np.sqrt(mean_squared_error(y_test, y_pred_test_linearr))
test_r2_linearr = r2_score(y_test, y_pred_test_linearr)




In [ ]:
ergebnis_linearr=[
    {"Modell" : modell_linearr, "Metrik": "MAE", "Training" : train_mae_linearr, "Test":test_mae_linearr},
    {"Modell" : modell_linearr, "Metrik": "RMSE", "Training" : train_rmse_linarr,  "Test":test_rmse_linearr},
    {"Modell" : modell_linearr, "Metrik": "R²", "Training" : train_r2_linearr,  "Test":test_r2_linearr}
     ]
ergebnis_linearr_df=pd.DataFrame(ergebnis_linearr)

In [ ]:
print(ergebnis_linearr_df.to_string(index=False))

In [ ]:
print("Model Evaluation")

print(f"{'Metrik':<15} {'Training':<15} {'Test':<15}")
print(f"{'MAE':<15} {train_mae_linearr:<15.2f} {test_mae_linearr:<15.2f}")
print(f"{'RMSE':<15} {train_rmse_linarr:<15.2f} {test_rmse_linearr:<15.2f}")
print(f"{'R²':<15} {train_r2_linearr:<15.4f} {test_r2_linearr:<15.4f}")

## Random Forest Modell

In [ ]:
from sklearn.ensemble import RandomForestRegressor



In [ ]:
# Features für Random Forest
features_rf = ["segment_id","stunde", "wochentag", "wochenende", "temperatur", "niederschlag", "biketotal_corrected", "cartotal_corrected", "DOY"]

X_rf = df_modellvariablen[features_rf]
y_rf = df_modellvariablen["pedtotal_corrected"]

# Train/Test Split
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf, y_rf, test_size=0.2, random_state=42
)

In [ ]:
# Random Forest trainieren
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_rf, y_train_rf)


In [ ]:
#Vorhersagen machen
y_pred_train_randomf = rf_model.predict(X_train_rf)
y_pred_test_randomf = rf_model.predict(X_test_rf)

In [ ]:
# Modell Random Forest evaluieren
modell_randomf="Random Forest"

train_mae_randomf = mean_absolute_error(y_train_rf, y_pred_train_randomf)
train_rmse_randomf = np.sqrt(mean_squared_error(y_train_rf, y_pred_train_randomf))
train_r2_randomf = r2_score(y_train_rf, y_pred_train_randomf)

test_mae_randomf = mean_absolute_error(y_test_rf, y_pred_test_randomf)
test_rmse_randomf = np.sqrt(mean_squared_error(y_test_rf, y_pred_test_randomf))
test_r2_randomf = r2_score(y_test_rf, y_pred_test_randomf)


In [ ]:
#Zusammenfassung Random Forest:
ergebnis_randomf=[
    {"Modell" : modell_randomf, "Metrik": "MAE", "Training" : train_mae_randomf, "Test":test_mae_randomf},
    {"Modell" : modell_randomf, "Metrik": "RMSE", "Training" : train_rmse_randomf,  "Test":test_rmse_randomf},
    {"Modell" : modell_randomf, "Metrik": "R²", "Training" : train_r2_randomf,  "Test":test_r2_randomf}
     ]
ergebnis_randomf_df=pd.DataFrame(ergebnis_randomf)

In [ ]:
print(ergebnis_randomf_df.to_string(index=False))

In [ ]:
#todo löschen (auch: falsche Variablen)
print("\nRandom Forest")
print(f"{'Metrik':<15} {'Training':<15} {'Test':<15}")
print(f"{'MAE':<15} {best_train_mae:<15.2f} {best_test_mae:<15.2f}")
print(f"{'RMSE':<15} {best_train_rmse:<15.2f} {best_test_rmse:<15.2f}")
print(f"{'R²':<15} {best_train_r2:<15.4f} {best_test_r2:<15.4f}")

## Hyperparameter Tuning mit 5-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import GridSearchCV

# Mit 5-Fold Cross Validation die besten Hyperparameter bestimmen:

# Hyperparameter-Grid definieren
param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8]
}

# Random Forest Base Model
#hier sind die Parameter drin die immer gleich sein sollen
rf_base = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# GridSearchCV mit 5-Fold Cross-Validation
# cv = 5 Folds
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error', #Metrik um die Random Forest Modell zu vergleichen
    verbose=2,
    n_jobs=-1
)

# Trainieren des Modells TODO:evtl validierungsmenge
grid_search.fit(X_train_rf, y_train_rf)

In [ ]:
print("\nBeste Hyperparameter:")
print(grid_search.best_params_)

#TODO: nochmal trainieren mit Testmenge

In [ ]:
# Modell mit besten Parametern evaluieren
modell_randomf_best="Random Forest mit optimierten Hyperparametern"
best_rf_model = grid_search.best_estimator_

y_pred_train_rf_bestestimator = best_rf_model.predict(X_train_rf)
y_pred_test_rf_bestestimator = best_rf_model.predict(X_test_rf)

best_train_mae = mean_absolute_error(y_train_rf, y_pred_train_rf_bestestimator)
best_train_rmse = np.sqrt(mean_squared_error(y_train_rf, y_pred_train_rf_bestestimator))
best_train_r2 = r2_score(y_train_rf, y_pred_train_rf_bestestimator)

best_test_mae = mean_absolute_error(y_test_rf, y_pred_test_rf_bestestimator)
best_test_rmse = np.sqrt(mean_squared_error(y_test_rf, y_pred_test_rf_bestestimator))
best_test_r2 = r2_score(y_test_rf, y_pred_test_rf_bestestimator)

print("\nRandom Forest (nach Hyperparameter-Tuning)")
print(f"{'Metrik':<15} {'Training':<15} {'Test':<15}")
print(f"{'MAE':<15} {best_train_mae:<15.2f} {best_test_mae:<15.2f}")
print(f"{'RMSE':<15} {best_train_rmse:<15.2f} {best_test_rmse:<15.2f}")
print(f"{'R²':<15} {best_train_r2:<15.4f} {best_test_r2:<15.4f}")

In [ ]:
#Zusammenfassung Random Forest:
ergebnis_randomf_best=[
    {"Modell" : modell_randomf_best, "Metrik": "MAE", "Training" : best_train_mae, "Test":best_test_mae},
    {"Modell" : modell_randomf_best, "Metrik": "RMSE", "Training" : best_train_rmse,  "Test":best_test_rmse},
    {"Modell" : modell_randomf_best, "Metrik": "R²", "Training" : best_train_r2,  "Test":best_test_r2}
     ]
ergebnis_randomfbest_df=pd.DataFrame(ergebnis_randomf_best)

# Modellevaluation Ende


In [ ]:
# Übersicht über die Testmetriken aller Modelle
evaluation_gesamt= pd.concat([ergebnis_baseline_df, ergebnis_linearr_df,ergebnis_randomf_df, ergebnis_randomfbest_df])
print(evaluation_gesamt.to_string(index=False))